<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2005%20-%202026-04-27%20-%20NumPy/clase_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 5 · NumPy

Hoy tocamos NumPy. Es raro al principio porque vienes de listas Python, y arrays son distintos — más rápidos, más restrictivos, y esa diferencia importa cuando procesas un millón de filas.

El corazón de NumPy es simple: **broadcasting**. La idea es que si quieres normalizar 1.000.000 de puntos, no escribas un bucle. NumPy entiende las dimensiones y replica internamente. Una línea. Eso es lo que verás hoy.

> **Hoy haces** · Los tres ejercicios sobre arrays, broadcasting e indexación booleana, las dos pausas y el checkpoint. ~2h30.
>
> **Entrega** · El notebook ejecutado al repo del equipo antes de la próxima clase.

In [ ]:
# --- Setup del entorno ---
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Estilo visual
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

print("Setup completo")
print(f"numpy {np.__version__} · pandas {pd.__version__}")

## Primero, el resultado

¿Cómo centramos y escalamos 1.000.000 de puntos en 0.3 segundos? La respuesta: vectorización + broadcasting. Veamos qué pasa cuando lo hacemos bien:

In [ ]:
# Datos simulados: 1M de puntos en 3D
np.random.seed(42)
datos_crudos = np.random.randn(1_000_000, 3) * 100 + 50

print(f"Shape original: {datos_crudos.shape}")
print(f"Primeras filas:\n{datos_crudos[:3]}")
print(f"Media por columna: {datos_crudos.mean(axis=0).round(2)}")
print(f"Std por columna: {datos_crudos.std(axis=0).round(2)}")

# Normalización con UNA LÍNEA (broadcast)
datos_normalizados = (datos_crudos - datos_crudos.mean(axis=0)) / datos_crudos.std(axis=0)

print(f"\nDespués de normalizar:")
print(f"Media: {datos_normalizados.mean(axis=0).round(6)}")
print(f"Std: {datos_normalizados.std(axis=0).round(6)}")

Eso que acaba de pasar: restar un vector (3 valores) de una matriz (1M × 3) y dividir por otro vector. NumPy lo hace sin bucles. Sin copia de memoria para cada fila. Sin Python creando listas intermedias. Solo operaciones C compiladas.

Eso es broadcasting.

## Construir arrays con formas específicas

Antes de hacer operaciones, tienes que saber crear los arrays. Hay funciones para zeros, ones, random. Prueba esto — completa los blancos:

In [ ]:
# Completa los huecos con la función correcta

# Vector de 5 ceros
vec = np.___((5,))

# Matriz 3x4 de unos
mat = np.___((3, 4))

# Tensor 2x2x3 con valores aleatorios uniformes [0,1)
tens = np.___((2, 2, 3))

print(f"Vec shape: {vec.shape}, dtype: {vec.dtype}")
print(f"Mat shape: {mat.shape}, dtype: {mat.dtype}")
print(f"Tens shape: {tens.shape}, dtype: {tens.dtype}")

In [ ]:
# Validación
assert "vec" in dir(), "La variable vec no existe"
assert vec.shape == (5,), f"vec debe tener shape (5,), tienes {vec.shape}"
assert mat.shape == (3, 4), f"mat debe tener shape (3, 4), tienes {mat.shape}"
assert tens.shape == (2, 2, 3), f"tens debe tener shape (2, 2, 3), tienes {tens.shape}"
print("✓ Ejercicio completado")

In [ ]:
# %% [solution]
# vec = np.zeros((5,))
# mat = np.ones((3, 4))
# tens = np.random.rand(2, 2, 3)

¿Por qué NumPy distingue entre shape `(5,)` y `(1, 5)`? Parece lo mismo pero no lo es. El primero es un vector 1D, el segundo es una matriz 1×5. En broadcasting, eso importa — una fila se copia diferente que un columna. Piénsalo un momento.

## Filtrado booleano en datos numéricos

Ahora con un caso real. Tienes precios y stock de 100 productos. Quieres los productos caros (> percentil 75) pero con bajo stock (< mediana). Esto es lo que hace un vendedor cuando revisa inventario.

In [ ]:
np.random.seed(42)
precios = np.random.uniform(10, 500, size=100)
stock = np.random.randint(1, 100, size=100)

p75 = np._____(precios, 75)
stock_mediana = np.median(stock)

# Indexación booleana: CAROS Y bajo stock
mask = (precios > p75) & (stock < stock_mediana)
productos_objetivo = precios[mask]

print(f"Percentil 75 de precios: ${p75:.2f}")
print(f"Mediana de stock: {stock_mediana:.0f} unidades")
print(f"Productos caros + bajo stock: {len(productos_objetivo)}")
print(f"Rango de precios objetivo: ${productos_objetivo.min():.2f} — ${productos_objetivo.max():.2f}")

In [ ]:
# Validación
assert "productos_objetivo" in dir(), "Variable no definida"
assert len(productos_objetivo) > 0, "Deberías encontrar al menos 1 producto"
assert (productos_objetivo > p75).all(), "Todos deben ser > percentil 75"
print("✓ Ejercicio completado")

In [ ]:
# %% [solution]
# p75 = np.percentile(precios, 75)

¿Cuántos productos encontraste? La lógica que usaste — buscar en dos dimensiones simultáneamente — es exactamente lo que hace data science en serio. Si llamamos "estrategia" a enfocarse en productos de alto valor con bajo inventario, ¿qué otra data agregarías para mejorar la decisión?

## Broadcasting + agregaciones

Esto es más complejo. Tienes una matriz de distancias (10×10) entre 10 puntos, y quieres identificar si hay clusters. Sin skeleton. El enunciado es claro: crea una matriz de distancias, identifica los puntos con distancia media baja (candidatos a centros de cluster).

In [ ]:
np.random.seed(42)
# Tu código aquí
# Crea matrix_distancias (10, 10) con np.random.normal(5, 2, (10, 10))
# Calcula mean_por_fila con axis=?
# Encuentra percentil_25
# Filtra puntos candidatos usando indexación booleana

# Cuando lo completes, deberías tener:
# print("Matriz shape:", matrix_distancias.shape)
# print("Distancia media por fila:", mean_por_fila)
# print("Puntos candidatos a centros:", puntos_candidatos)

In [ ]:
# Criterios de aceptación
assert "matrix_distancias" in dir() and matrix_distancias.shape == (10, 10)
assert "mean_por_fila" in dir() and mean_por_fila.shape == (10,)
assert "puntos_candidatos" in dir() and len(puntos_candidatos) > 0
print("✓ Ejercicio completado")

In [ ]:
# %% [solution]
# np.random.seed(42)
# matrix_distancias = np.random.normal(loc=5, scale=2, size=(10, 10))
# mean_por_fila = matrix_distancias.mean(axis=1)
# percentil_25 = np.percentile(mean_por_fila, 25)
# puntos_candidatos = np.where(mean_por_fila < percentil_25)[0]

## La verdad sobre velocidad

Ahora el benchmark. Porque todo esto importa cuando los datos son grandes.

In [ ]:
import time

def suma_bucle(arr):
    """Suma los elementos con un bucle Python."""
    total = 0.0
    for v in arr:
        total += v
    return total

# Preparar datos
np.random.seed(42)
N = 100_000
x = np.random.randn(N)

# Benchmark suma
t0 = time.time()
suma_python = suma_bucle(x)
t_python = time.time() - t0

t0 = time.time()
suma_numpy = x.sum()
t_numpy = time.time() - t0

print(f"Suma (N={N}):")
print(f"  Python:  {t_python*1000:.2f} ms")
print(f"  NumPy:   {t_numpy*1000:.3f} ms")
print(f"  Speedup: {t_python/t_numpy:.0f}x")

Ese factor 100—1000× es la razón por la que NumPy existe. Y cuando procesas millones de filas, eso no es un detalle — es la diferencia entre 10 minutos y 1 segundo.

## Checkpoint

Antes de seguir, verifiquemos que esto está claro:

1. **Broadcasting:** mecanismo que permite sumar arrays de shapes distintos sin bucles explícitos.
2. **axis=0** agrega filas, **axis=1** agrega columnas.
3. **Vectorización** es 100—1000× más rápido que bucles Python.

Si algo no quedó claro, ahora es mejor volver atrás que la próxima clase asuma que esto está sólido.

In [ ]:
# Validación automática
assert "vec" in dir() and vec.shape == (5,), "Reinicia y completa ejercicio 1"
print("Checkpoint ✓ — puedes continuar")

## Cómo piensa NumPy

```
Datos → Array NumPy → Operaciones vectorizadas
             ↓
       Broadcasting automático
             ↓
       Agregaciones (sum, mean, std)
             ↓
       Filtrado booleano (sin bucles)
             ↓
       Resultado (100—1000× más rápido)
```

## Para recordar

- **Arrays NumPy** son homogéneos, contiguos y rápidos vs. listas Python.
- **Broadcasting** permite operaciones entre shapes distintos sin bucles.
- **Agregaciones** con `axis=0` (filas) y `axis=1` (columnas).
- **Vectorización:** 100—1000× más rápido + código más limpio.
- **Indexación booleana:** `arr[arr > 5]` es tu amiga.

## Mañana

Veremos Pandas I — cargar datos reales, explorarlos y hacer selecciones. NumPy seguirá viviendo debajo, pero raramente lo tocarás directamente.

Para preparar:
- Relee la sección de broadcasting en el HTML si te quedó difuso.
- Practica creando arrays raros: `(1, 1, 5)`, `(3, 1)`, etc. Entiende por qué broadcast funciona con unos pero no con otros.

## Referencias

- [NumPy Absolute Beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) — empieza aquí.
- [Broadcasting Explained](https://numpy.org/doc/stable/user/basics.broadcasting.html) — cómo NumPy replica dimensiones.
- [Ten NumPy Tips for Data Science](https://towardsdatascience.com/ten-numpy-tips-for-data-science-b29ef6c5c1f5) — casos reales.

## Reflexión

En tus propias palabras, explícale a un compañero **por qué NumPy es más rápido que Python puro** en máximo 3 oraciones. Spoiler: no es porque "NumPy es mágico". Hay razones técnicas reales.

> **Recordatorio** · Notebook ejecutado al repo antes de la próxima sesión.

---

## Para tu equipo de proyecto

**Preparación para Pandas I:**

- Descarguen el dataset de su caso (CSV, Excel o API).
- Reporten:
  - Shape (filas × columnas)
  - Tipos (`dtypes`)
  - % nulos por columna
  - Primeras 5 filas
  - 1 observación sorprendente

**Entregable:** `proyecto_dataset_inicial.ipynb` en su repo.